## **Mount Google Drive**
This cell mounts your Google Drive to the Colab environment. This allows the notebook to access files stored in your Google Drive, which is essential for loading the dataset and saving processed results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **Import Necessary Libraries**
This cell imports all the Python libraries required for the notebook. These include `pandas` for data manipulation, `seaborn` and `matplotlib.pyplot` for visualization, `numpy` for numerical operations, `nibabel` for NIfTI image handling, `glob` for file path matching, `tensorflow.keras.utils` for `to_categorical` (one-hot encoding), `tifffile` for TIFF image handling, `MinMaxScaler` from `sklearn.preprocessing` for data scaling, `os` and `shutil` for file system operations, and `sklearn.model_selection` for splitting data.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import glob
from tensorflow.keras.utils import to_categorical
from tifffile import imwrite
from sklearn.preprocessing import MinMaxScaler
import os
import random
import shutil
from sklearn.model_selection import train_test_split

## **Initialize Scaler and Define Base Paths**
This cell initializes a `MinMaxScaler` object, which will be used to normalize image pixel intensities. It also defines several important file paths: `BRAIN_TUMOR_DATA_BASE_PATH` points to the raw BraTS2020 training data, `PROCESSED_DATA_BASE_PATH` to where 3-channel processed data will be stored, and `SPLIT_DATA_BASE_PATH` for the final 128x128x128 split data.

In [ ]:
scaler = MinMaxScaler()

# Base path for the raw BraTS2020 training data
BRAIN_TUMOR_DATA_BASE_PATH = 'your/path/to/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

# Base path for the processed 3-channel data (input_data_3channels)
PROCESSED_DATA_BASE_PATH = 'your/path/to/processed_data_3channels'

# Base path for the final 128x128x128 split data (input_data_128)
SPLIT_DATA_BASE_PATH = 'your/path/to/split_data_128'

## **Load Survival Information CSV**
This cell loads the `survival_info.csv` file into a pandas DataFrame named `df`. This CSV likely contains metadata related to the BraTS2020 dataset, such as patient survival information, which could be used for further analysis or model training if required.

In [ ]:
name = "your/path/to/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/survival_info.csv"
df = pd.read_csv(name)

## **Load and Initial Visualization of a Single MRI Case**
This cell loads a specific patient's (ID 354) MRI images (Flair, T1, T1ce, T2) and its corresponding segmentation mask. It then scales the image data using `MinMaxScaler` and preprocesses the mask by reassigning the value 4 to 3. Finally, it visualizes random slices of these images and the mask to provide an initial understanding of the raw data structure and content.

In [ ]:
TRAIN_DATASET_PATH = 'your/path/to/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

test_image_flair=nib.load(TRAIN_DATASET_PATH + '/BraTS20_Training_354/BraTS20_Training_354_flair.nii').get_fdata()
# Scalers are applied to 1D, so reshape and then reshape back to original shape.
test_image_flair=scaler.fit_transform(test_image_flair.reshape(-1, test_image_flair.shape[-1])).reshape(test_image_flair.shape)


test_image_t1=nib.load(TRAIN_DATASET_PATH + '/BraTS20_Training_354/BraTS20_Training_354_t1.nii').get_fdata()
test_image_t1=scaler.fit_transform(test_image_t1.reshape(-1, test_image_t1.shape[-1])).reshape(test_image_t1.shape)

test_image_t1ce=nib.load(TRAIN_DATASET_PATH + '/BraTS20_Training_354/BraTS20_Training_354_t1ce.nii').get_fdata()
test_image_t1ce=scaler.fit_transform(test_image_t1ce.reshape(-1, test_image_t1ce.shape[-1])).reshape(test_image_t1ce.shape)

test_image_t2=nib.load(TRAIN_DATASET_PATH + '/BraTS20_Training_354/BraTS20_Training_354_t2.nii').get_fdata()
test_image_t2=scaler.fit_transform(test_image_t2.reshape(-1, test_image_t2.shape[-1])).reshape(test_image_t2.shape)

test_mask=nib.load(TRAIN_DATASET_PATH + '/BraTS20_Training_354/BraTS20_Training_354_seg.nii').get_fdata()
test_mask=test_mask.astype(np.uint8)

print(np.unique(test_mask))  # 0, 1, 2, 4 (Needs re-encoding to 0, 1, 2, 3)
test_mask[test_mask==4] = 3  # Reassign mask values from 4 to 3
print(np.unique(test_mask))

import random
n_slice=random.randint(0, test_mask.shape[2])

plt.figure(figsize=(12, 8))

plt.subplot(231)
plt.imshow(test_image_flair[:,:,n_slice], cmap='gray')
plt.title('Image flair')
plt.subplot(232)
plt.imshow(test_image_t1[:,:,n_slice], cmap='gray')
plt.title('Image t1')
plt.subplot(233)
plt.imshow(test_image_t1ce[:,:,n_slice], cmap='gray')
plt.title('Image t1ce')
plt.subplot(234)
plt.imshow(test_image_t2[:,:,n_slice], cmap='gray')
plt.title('Image t2')
plt.subplot(235)
plt.imshow(test_mask[:,:,n_slice])
plt.title('Mask')
plt.show()

## **Combine and Crop Modalities for Visualization**
This cell combines the Flair, T1ce, and T2 MRI images into a single multichannel NumPy array. It then crops both the combined image and the mask to a `128x128x128` size. This step is part of exploring how the data looks after initial combination and cropping, which is often a necessary preprocessing step for model input. The cropped images and mask are visualized to confirm the operation.

In [ ]:
# Combining all 4 images into a 4-channel numpy array.
# Flair, T1CE, and T2 modalities contain the most information
# Combine t1ce, t2, and flair into a single multichannel image

combined_x = np.stack([test_image_flair, test_image_t1ce, test_image_t2], axis=3)

# Crop to a size divisible by 64 for extracting 64x64x64 patches.
# Cropping along x, y, and z dimensions.
#combined_x=combined_x[24:216, 24:216, 13:141]

combined_x=combined_x[56:184, 56:184, 13:141] # Crop to 128x128x128x4

# Apply the same cropping to the mask.
test_mask = test_mask[56:184, 56:184, 13:141]

n_slice=random.randint(0, test_mask.shape[2])
plt.figure(figsize=(12, 8))

plt.subplot(221)
plt.imshow(combined_x[:,:,n_slice, 0], cmap='gray')
plt.title('Image flair')
plt.subplot(222)
plt.imshow(combined_x[:,:,n_slice, 1], cmap='gray')
plt.title('Image t1ce')
plt.subplot(223)
plt.imshow(combined_x[:,:,n_slice, 2], cmap='gray')
plt.title('Image t2')
plt.subplot(224)
plt.imshow(test_mask[:,:,n_slice])
plt.title('Mask')
plt.show()

## **Comprehensive Validation of Processed Data**
This cell rigorously validates the NumPy arrays that were generated and saved in the previous processing step. It verifies that the number of image and mask files match, that their IDs are consistent, and then samples a few cases. For each sampled case, it checks the image and mask shapes, confirms that the masks are correctly one-hot encoded, and calculates the ratio of different labels within the mask. Visualizations are also provided to allow for a qualitative assessment of the processed data.

In [ ]:
test_mask = to_categorical(test_mask, num_classes=4)

In [ ]:
!ls 'BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

In [ ]:
# Image lists
#t1_list = sorted(glob.glob('BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/*/*t1.nii'))

t2_list = sorted(glob.glob("your/path/to/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/*/*t2.nii"))
t1ce_list = sorted(glob.glob('your/path/to/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/*/*t1ce.nii'))
flair_list = sorted(glob.glob('your/path/to/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/*/*flair.nii'))
mask_list = sorted(glob.glob('your/path/to/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/*/*seg.nii'))

## **Verify Counts of Image Modalities**
This cell simply prints the number of files found for each MRI modality (T2, T1ce, Flair, and masks). This provides a quick check to ensure that the dataset has a consistent number of files across all modalities, which is important for data integrity.

In [ ]:
print(len(t2_list))
print(len(t1ce_list))
print(len(flair_list))
print(len(mask_list))

## **Check for Missing Data Files**
This cell performs an integrity check across the dataset. It defines a utility function to extract unique patient IDs from filenames and then compares the IDs across all modalities (T2, T1ce, Flair, and mask). It reports any cases where a patient ID is found to be missing a file for any of the modalities, indicating potential data incompleteness.

In [ ]:
import os
def clean_ids(file_list, suffix):
    return set(os.path.basename(f).replace(suffix, "") for f in file_list)

t2_ids = clean_ids(t2_list, "_t2.nii")
t1ce_ids = clean_ids(t1ce_list, "_t1ce.nii")
flair_ids = clean_ids(flair_list, "_flair.nii")
mask_ids = clean_ids(mask_list, "_seg.nii")

all_ids = t2_ids | t1ce_ids | flair_ids | mask_ids

for case in sorted(all_ids):
    if case not in t2_ids:
        print(case, "missing t2")
    if case not in t1ce_ids:
        print(case, "missing t1ce")
    if case not in flair_ids:
        print(case, "missing flair")
    if case not in mask_ids:
        print(case, "missing mask")

## **Batch Process and Save MRI Data**
This cell iterates through all the patient cases. For each case, it loads the T2, T1ce, Flair, and mask NIfTI files, scales the image data, and preprocesses the mask by reassigning the value 4 to 3. It then combines the image modalities into a 4-channel array and crops both the image and mask to `128x128x128`. A crucial step is the `if` condition: it checks if the mask contains at least 1% useful volume (non-zero labels). If it does, the combined image and one-hot encoded mask are saved as NumPy arrays to the `input_data_3channels` directory; otherwise, the case is considered 'useless' and skipped.

In [ ]:
# Each volume generates 18 64x64x64x4 sub-volumes.
# Total 369 volumes = 6642 sub volumes

for img in range(len(t2_list)):   # Using t2_list as all lists are of the same size
    print("Now preparing image and masks number: ", img)

    temp_image_t2=nib.load(t2_list[img]).get_fdata()
    temp_image_t2=scaler.fit_transform(temp_image_t2.reshape(-1, temp_image_t2.shape[-1])).reshape(temp_image_t2.shape)

    temp_image_t1ce=nib.load(t1ce_list[img]).get_fdata()
    temp_image_t1ce=scaler.fit_transform(temp_image_t1ce.reshape(-1, temp_image_t1ce.shape[-1])).reshape(temp_image_t1ce.shape)

    temp_image_flair=nib.load(flair_list[img]).get_fdata()
    temp_image_flair=scaler.fit_transform(temp_image_flair.reshape(-1, temp_image_flair.shape[-1])).reshape(temp_image_flair.shape)

    temp_mask=nib.load(mask_list[img]).get_fdata()
    temp_mask=temp_mask.astype(np.uint8)
    temp_mask[temp_mask==4] = 3  # Reassign mask values from 4 to 3
    #print(np.unique(temp_mask))


    temp_combined_images = np.stack([temp_image_flair, temp_image_t1ce, temp_image_t2], axis=3)

    # Crop to a size divisible by 64 for extracting 64x64x64 patches.
    # Cropping along x, y, and z dimensions.
    temp_combined_images=temp_combined_images[56:184, 56:184, 13:141]
    temp_mask = temp_mask[56:184, 56:184, 13:141]

    val, counts = np.unique(temp_mask, return_counts=True)

    if (1 - (counts[0]/counts.sum())) > 0.01:  # At least 1% useful volume with non-zero labels
        print("Save Me")
        temp_mask= to_categorical(temp_mask, num_classes=4)
        np.save('/you/project/path/BraTS2020_TrainingData/input_data_3channels/images/image_'+str(img)+'.npy',temp_combined_images)
        np.save('/you/project/path/BraTS2020_TrainingData/input_data_3channels/masks/mask_'+str(img)+'.npy',temp_mask)

    else:
        print("I am useless")

In [ ]:
import numpy as np
import os
import random
import matplotlib.pyplot as plt

image_path = 'your/project/path/BraTS2020_TrainingData/input_data_3channels/images/'
mask_path  = 'your/project/path/BraTS2020_TrainingData/input_data_3channels/masks/'

image_files = sorted([f for f in os.listdir(image_path) if f.endswith('.npy')])
mask_files  = sorted([f for f in os.listdir(mask_path) if f.endswith('.npy')])

print("Total saved images:", len(image_files))
print("Total saved masks :", len(mask_files))
assert len(image_files) == len(mask_files), "Image/mask file count mismatch!"

# (Optional) Strict check: ensure corresponding IDs in filenames.
def extract_id(fname):
    # image_12.npy / mask_12.npy -> 12
    base = os.path.splitext(fname)[0]
    return base.split('_')[-1]

for i in range(len(image_files)):
    if extract_id(image_files[i]) != extract_id(mask_files[i]):
        raise ValueError(f"Filename ID mismatch at sorted index {i}: {image_files[i]} vs {mask_files[i]}")
print("Filename IDs match ✅")

# Randomly sample 10 cases (or all if fewer than 10).
k = min(10, len(image_files))
indices = random.sample(range(len(image_files)), k)

print("\nRandom check indices:", indices)

for n, idx in enumerate(indices, 1):
    img_file = image_files[idx]
    msk_file = mask_files[idx]
    img = np.load(os.path.join(image_path, img_file))
    mask = np.load(os.path.join(mask_path, msk_file))

    # Shape check
    ok_shape = (img.shape[:3] == mask.shape[:3]) and (img.shape[-1] == 3) and (mask.shape[-1] == 4)

    # One-hot check: each voxel's channel sum should be 1 (tolerance for minor errors).
    mask_sum = mask.sum(axis=-1)
    onehot_ok = np.all((mask == 0) | (mask == 1)) and np.all((mask_sum == 1))

    # Class ratio
    labels = np.argmax(mask, axis=-1)
    vals, counts = np.unique(labels, return_counts=True)
    ratio = {int(v): float(c) / counts.sum() for v, c in zip(vals, counts)}

    print(f"\n[{n}/{k}] {img_file}  <->  {msk_file}")
    print("  Image shape:", img.shape, " Mask shape:", mask.shape)
    print("  Shape OK   :", "✅" if ok_shape else "❌")
    print("  One-hot OK :", "✅" if onehot_ok else "❌")
    print("  Label ratio:", ratio)

    # Visualization: taking the middle slice (you can also use random.randint(0, img.shape[2]-1)).
    z = img.shape[2] // 2

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.title(f"FLAIR (z={z})")
    plt.imshow(img[:, :, z, 0], cmap="gray")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Overlay (mask argmax)")
    plt.imshow(img[:, :, z, 0], cmap="gray")
    plt.imshow(labels[:, :, z], alpha=0.4)
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import splitfolders

## **Explanation of Training and Validation Split Validation (Text Cell)**
This markdown cell (`6213fd9c`) serves as a descriptive header and high-level explanation for the code cell that immediately follows it. It outlines the purpose of the subsequent code block, which is to thoroughly validate the training and validation sets created previously, detailing the checks performed such as file count matching, ID consistency, one-hot encoding integrity, leak checks, and ratio analysis.

In [ ]:
input_folder = '/your/project/path/BraTS2020_TrainingData/input_data_3channels'
output_folder = '/your/project/path/BraTS2020_TrainingData/input_data_128'

# Remove the output folder if it exists to ensure a clean retry
if os.path.exists(output_folder):
    print(f"Removing existing output folder: {output_folder}")
    shutil.rmtree(output_folder)
    print("Output folder removed.")

# --- Start of new logic for manual splitting ---

# 1. Get all unique IDs from the source image files
source_image_path = os.path.join(input_folder, 'images')
all_image_files = sorted([f for f in os.listdir(source_image_path) if f.endswith('.npy')])

def extract_id(fname):
    # Extracts the numeric ID from filenames like 'image_12.npy'
    return os.path.splitext(fname)[0].split('_')[-1]

all_ids = [extract_id(f) for f in all_image_files]

# 2. Split the IDs into training and validation sets
train_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42) # 80/20 split

print(f"Total unique IDs found: {len(all_ids)}")
print(f"Number of IDs for training: {len(train_ids)}")
print(f"Number of IDs for validation: {len(val_ids)}")

# 3. Create target directories for train/val splits
train_img_dir = os.path.join(output_folder, 'train', 'images')
train_mask_dir = os.path.join(output_folder, 'train', 'masks')
val_img_dir = os.path.join(output_folder, 'val', 'images')
val_mask_dir = os.path.join(output_folder, 'val', 'masks')

os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(train_mask_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_mask_dir, exist_ok=True)

# 4. Copy files for the training set
print("Copying training files...")
for sid in train_ids:
    shutil.copy(
        os.path.join(input_folder, 'images', f"image_{sid}.npy"),
        os.path.join(train_img_dir, f"image_{sid}.npy")
    )
    shutil.copy(
        os.path.join(input_folder, 'masks', f"mask_{sid}.npy"),
        os.path.join(train_mask_dir, f"mask_{sid}.npy")
    )

# 5. Copy files for the validation set
print("Copying validation files...")
for sid in val_ids:
    shutil.copy(
        os.path.join(input_folder, 'images', f"image_{sid}.npy"),
        os.path.join(val_img_dir, f"image_{sid}.npy")
    )
    shutil.copy(
        os.path.join(input_folder, 'masks', f"mask_{sid}.npy"),
        os.path.join(val_mask_dir, f"mask_{sid}.npy")
    )

print("File splitting and copying complete.")

## **Validate Training and Validation Split**
This cell performs a thorough validation of the training and validation sets created previously. It defines helper functions to list `.npy` files and extract IDs. For both the training and validation sets, it checks if image and mask file counts match, if their IDs are consistent, and then samples a few cases to verify image and mask shapes, one-hot encoding integrity, and label ratios. A critical 'leak check' ensures there's no overlap between training and validation IDs to prevent data leakage. Finally, it reports the ratio of training data to the total dataset. Visualizations of sampled images and masks are included to visually confirm the split.

In [ ]:
output_folder = '/your/project/path/BraTS2020_TrainingData/input_data_128'

def list_npy(folder):
    return sorted([f for f in os.listdir(folder) if f.endswith('.npy')])

def extract_id(fname):
    # image_12.npy / mask_12.npy -> 12
    return os.path.splitext(fname)[0].split('_')[-1]

def check_split(split_name, k=5, show=True):
    img_dir = os.path.join(output_folder, split_name, 'images')
    msk_dir = os.path.join(output_folder, split_name, 'masks')

    assert os.path.exists(img_dir), f"Missing: {img_dir}"
    assert os.path.exists(msk_dir), f"Missing: {msk_dir}"

    imgs = list_npy(img_dir)
    msks = list_npy(msk_dir)

    print(f"\n=== {split_name.upper()} ===")
    print("Images:", len(imgs), "Masks:", len(msks))
    assert len(imgs) == len(msks), f"{split_name}: image/mask count mismatch!"

    img_ids = [extract_id(f) for f in imgs]
    msk_ids = [extract_id(f) for f in msks]

    # Check for consistent ID sets (order doesn't necessarily have to match).
    assert set(img_ids) == set(msk_ids), f"{split_name}: image IDs != mask IDs"
    print("ID sets match ✅")

    # Randomly sample k items.
    sample_ids = random.sample(list(set(img_ids)), min(k, len(set(img_ids))))
    print("Sample IDs:", sample_ids)

    for sid in sample_ids:
        img_file = f"image_{sid}.npy"
        msk_file = f"mask_{sid}.npy"

        img_path = os.path.join(img_dir, img_file)
        msk_path = os.path.join(msk_dir, msk_file)

        assert os.path.exists(img_path), f"Missing image: {img_path}"
        assert os.path.exists(msk_path), f"Missing mask : {msk_path}"

        img = np.load(img_path)
        mask = np.load(msk_path)

        ok_shape = (img.shape[:3] == mask.shape[:3]) and (img.shape[-1] == 3) and (mask.shape[-1] == 4)

        # One-hot check
        mask_sum = mask.sum(axis=-1)
        onehot_ok = np.all((mask == 0) | (mask == 1)) and np.all(mask_sum == 1)

        labels = np.argmax(mask, axis=-1)
        vals, counts = np.unique(labels, return_counts=True)
        ratio = {int(v): float(c)/counts.sum() for v, c in zip(vals, counts)}

        print(f"  ID {sid} | img {img.shape} mask {mask.shape} | shape {'✅' if ok_shape else '❌'} | onehot {'✅' if onehot_ok else '❌'} | ratio {ratio}")

        if show:
            z = img.shape[2] // 2
            plt.figure(figsize=(10,4))
            plt.subplot(1,2,1)
            plt.title(f"{split_name} FLAIR z={z} (ID={sid})")
            plt.imshow(img[:,:,z,0], cmap='gray'); plt.axis('off')

            plt.subplot(1,2,2)
            plt.title("Overlay")
            plt.imshow(img[:,:,z,0], cmap='gray')
            plt.imshow(labels[:,:,z], alpha=0.4); plt.axis('off')
            plt.tight_layout()
            plt.show()

    return set(img_ids)

# 1) Check train/validation sets separately
train_ids = check_split("train", k=5, show=True)
val_ids   = check_split("val",   k=5, show=True)

# 2) Check for no overlap between train/validation sets (prevent data leakage)
inter = train_ids.intersection(val_ids)
print("\n=== LEAK CHECK ===")
print("Train ∩ Val =", len(inter))
assert len(inter) == 0, f"Data leakage! Overlapped IDs: {list(inter)[:20]}"
print("No leakage ✅")

# 3) Check approximate 8/2 ratio (allowing minor errors)
total = len(train_ids) + len(val_ids)
train_ratio = len(train_ids) / total if total else 0
print("\n=== RATIO CHECK ===")
print(f"Total: {total} | Train: {len(train_ids)} | Val: {len(val_ids)} | Train ratio: {train_ratio:.3f}")